<a href="https://colab.research.google.com/github/shatrill/data_analytics/blob/main/session_3_sql_fundamentals_sqlite.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 3: Fundamental SQL Queries with SQLite

**Course:** Data Analytics Foundation (CompTIA Data+) | **CompTIA Domain:** 1.0 Data Concepts & Environments | 3.0 Data Analysis

> This notebook runs real SQL inside **Google Colab** using **SQLite**, a full database engine that ships with Python, so there is nothing to install. Every query you write here works almost word-for-word in SQL Fiddle, PostgreSQL, MySQL, SQL Server and Snowflake.

## 🗺️ Your 45-Minute Lab Map
| Part | What you will do | Approx. time |
|---|---|---|
| Setup | Create a database and a helper to run SQL | 4 min |
| Part 1 | Build the `employees` table (DDL + 6 records) | 5 min |
| Part 2 | SELECT, WHERE, ORDER BY, GROUP BY, HAVING with ✏️ TODOs | 16 min |
| Part 3 | Guided Example: grow the table and build a query clause by clause | 10 min |
| Part 4 | 🧩 Guided Challenge: the Regional Sales report | 10 min |

**How to use this notebook:** read each text cell, then run the code cell beneath it with **Shift + Enter**.
Cells marked **✏️ Your Turn** have blanks (`___`) for you to fill in. Each one has a hidden solution
you can reveal *after* you have tried it. Errors are normal and part of learning. Read the last line
of the error message, fix it, and run the cell again.

## Setup: A Database in One Line

- `sqlite3` is Python's built-in database engine.
- `pandas.read_sql_query()` runs a SQL query and returns the answer as a neat table.

We will also write a tiny helper function, `run()`, so every query is a single line.

In [3]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")   # a brand-new database that lives in memory

def run(sql):
    """Run a SELECT query and show the result as a table."""
    return pd.read_sql_query(sql, conn)

print("Database ready!")

Database ready!


## Part 1: Create the `employees` Table

**DDL (Data Definition Language)** defines the structure: table name, columns, data types and keys.
**DML (Data Manipulation Language)** adds or changes the rows.

`employee_id` is the **Primary Key (PK)**: the unique fingerprint of every row. It can never repeat and can never be empty.

In [ ]:
conn.executescript('''
DROP TABLE IF EXISTS employees;

CREATE TABLE employees (
    employee_id  INTEGER PRIMARY KEY,
    first_name   TEXT    NOT NULL,
    last_name    TEXT    NOT NULL,
    department   TEXT,
    salary       INTEGER,
    hire_date    TEXT,       -- stored as 'YYYY-MM-DD'
    city         TEXT
);

INSERT INTO employees VALUES
 (1, 'Ana',    'Rivera',   'Engineering', 95000, '2019-03-11', 'Rochester'),
 (2, 'Ben',    'Okafor',   'Marketing',   62000, '2021-07-01', 'Buffalo'),
 (3, 'Carla',  'Nguyen',   'Engineering', 105000,'2017-01-23', 'Rochester'),
 (4, 'Derek',  'Schmidt',  'Sales',       58000, '2022-09-15', 'Syracuse'),
 (5, 'Elena',  'Petrova',  'Marketing',   71000, '2018-05-30', 'Rochester'),
 (6, 'Farid',  'Haddad',   'Sales',       64000, '2020-11-02', 'Buffalo');
''')
run("SELECT * FROM employees")

,employee_id,first_name,last_name,department,salary,hire_date,city
0,1,Ana,Rivera,Engineering,95000,2019-03-11,Rochester
1,2,Ben,Okafor,Marketing,62000,2021-07-01,Buffalo
2,3,Carla,Nguyen,Engineering,105000,2017-01-23,Rochester
3,4,Derek,Schmidt,Sales,58000,2022-09-15,Syracuse
4,5,Elena,Petrova,Marketing,71000,2018-05-30,Rochester
5,6,Farid,Haddad,Sales,64000,2020-11-02,Buffalo


## Part 2: The Core Clauses

### 2.1 `SELECT` and `FROM`: "which columns, from which table?"
`SELECT *` means "all columns." It is fine for exploring, but in real work name the columns you need.

In [ ]:
run('''
SELECT first_name, last_name, department
FROM employees
''')

,first_name,last_name,department
0,Ana,Rivera,Engineering
1,Ben,Okafor,Marketing
2,Carla,Nguyen,Engineering
3,Derek,Schmidt,Sales
4,Elena,Petrova,Marketing
5,Farid,Haddad,Sales


In [ ]:
# TODO: show only the first_name and salary columns
run('''
SELECT ___, ___
FROM ___
''')

DatabaseError: Execution failed on sql '
SELECT ___, ___
FROM ___
': no such table: ___

<details>
<summary><b>💡 Reveal a possible solution</b> (try it yourself first!)</summary>

```python
run('''
SELECT first_name, salary
FROM employees
''')
```

</details>

### 2.2 `WHERE`: "only the rows that match"
Comparison operators: `=`, `>`, `<`, `>=`, `<=`, `<>` (not equal). Text goes in **single quotes**.

In [ ]:
run('''
SELECT first_name, department, salary
FROM employees
WHERE department = 'Engineering'
''')

In [ ]:
# TODO: find everyone who earns MORE than 65,000
run('''
SELECT first_name, salary
FROM employees
WHERE 65
''')

,first_name,salary
0,Ana,95000
1,Ben,62000
2,Carla,105000
3,Derek,58000
4,Elena,71000
5,Farid,64000


<details>
<summary><b>💡 Reveal a possible solution</b> (try it yourself first!)</summary>

```python
run('''
SELECT first_name, salary
FROM employees
WHERE salary > 65000
''')
```

</details>

### 2.3 `ORDER BY`: "sort the output"
`ASC` = smallest first (the default). `DESC` = largest first.

In [ ]:
run('''
SELECT first_name, salary
FROM employees
ORDER BY salary ASC
''')

,first_name,salary
0,Derek,58000
1,Ben,62000
2,Farid,64000
3,Elena,71000
4,Ana,95000
5,Carla,105000


In [ ]:
# TODO: list everyone in Rochester, ordered by hire_date (oldest hire first)
run('''
SELECT first_name, city, hire_date
FROM employees
WHERE city = "rochester"
ORDER BY hire_date ASC
''')

,first_name,city,hire_date


<details>
<summary><b>💡 Reveal a possible solution</b> (try it yourself first!)</summary>

```python
run('''
SELECT first_name, city, hire_date
FROM employees
WHERE city = 'Rochester'
ORDER BY hire_date ASC
''')
```

</details>

### 2.4 `GROUP BY` with aggregate functions
Aggregate functions collapse many rows into one number: `COUNT()`, `SUM()`, `AVG()`, `MIN()`, `MAX()`.
`GROUP BY` makes one bucket per value, then runs the aggregate inside each bucket.

**Rule of thumb:** every column in `SELECT` that is *not* inside an aggregate must appear in `GROUP BY`.

`AS` gives a column a friendly name (an **alias**).

In [ ]:
run('''
SELECT department,
       COUNT(*)            AS headcount,
       SUM(salary)         AS total_payroll,
       ROUND(AVG(salary))  AS avg_salary
FROM employees
GROUP BY department
''')

,department,headcount,total_payroll,avg_salary
0,Engineering,2,200000,100000.0
1,Marketing,2,133000,66500.0
2,Sales,2,122000,61000.0


In [ ]:
# TODO: count employees and find the highest salary in each city
run('''
SELECT city,
       ___(*)       AS headcount,
       ___(salary)  AS top_salary
FROM employees
GROUP BY ___
''')

DatabaseError: Execution failed on sql '
SELECT city,
       ___(*)       AS headcount,
       ___(salary)  AS top_salary
FROM employees
GROUP BY ___
': no such function: ___

<details>
<summary><b>💡 Reveal a possible solution</b> (try it yourself first!)</summary>

```python
run('''
SELECT city,
       COUNT(*)     AS headcount,
       MAX(salary)  AS top_salary
FROM employees
GROUP BY city
''')
```

</details>

### 2.5 `HAVING` vs. `WHERE`: the bouncer analogy
- **`WHERE`** is the bouncer at the front door. It checks **individual rows** *before* they are grouped.
- **`HAVING`** is the bouncer at the VIP room. It checks **whole groups** *after* `GROUP BY` has done its math.

⚠️ **CompTIA trap:** you cannot put an aggregate such as `AVG(salary)` inside `WHERE`. Use `HAVING`.

In [ ]:
# Departments whose AVERAGE salary is over 80,000
run('''
SELECT department, ROUND(AVG(salary)) AS avg_salary
FROM employees
GROUP BY department
HAVING AVG(salary) > 80000
''')

,department,avg_salary
0,Engineering,100000.0


In [ ]:
# TODO: show departments that have MORE THAN 1 employee AND a total payroll over 130,000
run('''
SELECT department, COUNT(*) AS headcount, SUM(salary) AS total_payroll
FROM employees
GROUP BY department
HAVING headcount > 1 AND total_payroll > 130000
''')

,department,headcount,total_payroll
0,Engineering,2,200000
1,Marketing,2,133000


<details>
<summary><b>💡 Reveal a possible solution</b> (try it yourself first!)</summary>

```python
run('''
SELECT department, COUNT(*) AS headcount, SUM(salary) AS total_payroll
FROM employees
GROUP BY department
HAVING COUNT(*) > 1 AND SUM(salary) > 130000
''')
```

</details>

## Part 3: Guided Example: Grow the Table, Then Build a Query Clause by Clause

### Step 1: The company hires six more people
`INSERT INTO ... VALUES` adds rows. Listing the column names first is the safest habit.

In [ ]:
conn.executescript('''
INSERT INTO employees (employee_id, first_name, last_name, department, salary, hire_date, city) VALUES
 (7,  'Grace',  'Kim',      'Engineering', 88000, '2023-02-14', 'Buffalo'),
 (8,  'Hector', 'Alvarez',  'Sales',       52000, '2024-01-08', 'Rochester'),
 (9,  'Imani',  'Brooks',   'Finance',     79000, '2019-08-19', 'Syracuse'),
 (10, 'Jonas',  'Lindqvist','Engineering', 112000,'2016-06-06', 'Rochester'),
 (11, 'Kira',   'Tanaka',   'Finance',     83000, '2021-03-29', 'Rochester'),
 (12, 'Luis',   'Moreno',   'Marketing',   59000, '2023-10-10', 'Syracuse');
''')
run("SELECT COUNT(*) AS total_employees FROM employees")

,total_employees
0,12


### Step 2: Watch the Primary Key protect us
What if someone tries to reuse `employee_id` 3? The database refuses. That is **relational integrity** at work.
`try / except` lets us catch the error politely instead of crashing.

In [ ]:
try:
    conn.execute("INSERT INTO employees VALUES (3, 'Duplicate', 'Person', 'Sales', 1, '2025-01-01', 'Nowhere')")
except sqlite3.IntegrityError as e:
    print("❌ The database blocked it:", e)

❌ The database blocked it: UNIQUE constraint failed: employees.employee_id


### Step 3: Answer a real question, one clause at a time
> **The VP of People asks:** *"For employees hired since 2018, which departments have an average salary above $70,000? Show headcount too, highest average first."*

We build the query in the same order SQL *thinks* about it:

`FROM` → `WHERE` → `GROUP BY` → `HAVING` → `SELECT` → `ORDER BY`

**3a. FROM + WHERE:** start with the rows we care about (hired 2018 or later). Dates in `YYYY-MM-DD` format sort correctly as text.

In [ ]:
run('''
SELECT first_name, department, salary, hire_date
FROM employees
WHERE hire_date >= '2018-01-01'
ORDER BY department
''')

,first_name,department,salary,hire_date
0,Ana,Engineering,95000,2019-03-11
1,Grace,Engineering,88000,2023-02-14
2,Imani,Finance,79000,2019-08-19
3,Kira,Finance,83000,2021-03-29
4,Ben,Marketing,62000,2021-07-01
5,Elena,Marketing,71000,2018-05-30
6,Luis,Marketing,59000,2023-10-10
7,Derek,Sales,58000,2022-09-15
8,Farid,Sales,64000,2020-11-02
9,Hector,Sales,52000,2024-01-08


**3b. Add GROUP BY:** one row per department, with headcount and average salary.

In [ ]:
run('''
SELECT department,
       COUNT(*)           AS headcount,
       ROUND(AVG(salary)) AS avg_salary
FROM employees
WHERE hire_date >= '2018-01-01'
GROUP BY department
''')

,department,headcount,avg_salary
0,Engineering,2,91500.0
1,Finance,2,81000.0
2,Marketing,3,64000.0
3,Sales,3,58000.0


**3c. Add HAVING:** keep only departments whose average is over 70,000.

In [ ]:
run('''
SELECT department,
       COUNT(*)           AS headcount,
       ROUND(AVG(salary)) AS avg_salary
FROM employees
WHERE hire_date >= '2018-01-01'
GROUP BY department
HAVING AVG(salary) > 70000
''')

,department,headcount,avg_salary
0,Engineering,2,91500.0
1,Finance,2,81000.0


**3d. Add ORDER BY:** highest average first. Done!

In [ ]:
run('''
SELECT department,
       COUNT(*)           AS headcount,
       ROUND(AVG(salary)) AS avg_salary
FROM employees
WHERE hire_date >= '2018-01-01'
GROUP BY department
HAVING AVG(salary) > 70000
ORDER BY avg_salary DESC
''')

,department,headcount,avg_salary
0,Engineering,2,91500.0
1,Finance,2,81000.0


**💬 Why build step by step?** If the final answer looks wrong, you can run each stage separately to find exactly where it went off track. Professional analysts debug SQL this way.

## Part 4: 🧩 Guided Challenge: The Regional Sales Report

The Sales Director just gave you a `sales` table. Run the setup cell, then answer the five questions.

Two new helpers you may want:
- `IN ('A', 'B')` matches any value in a list, for example `WHERE region IN ('East', 'West')`.
- `LIMIT n` returns only the first *n* rows of the result, which is handy for "Top 3" questions.

In [9]:
conn.executescript('''
DROP TABLE IF EXISTS sales;
CREATE TABLE sales (
    sale_id     INTEGER PRIMARY KEY,
    employee_id INTEGER,
    region      TEXT,
    product     TEXT,
    amount      REAL,
    sale_date   TEXT
);
INSERT INTO sales VALUES
 (1, 4, 'East',  'Laptop',  1200.00, '2025-01-05'),
 (2, 6, 'West',  'Monitor',  300.00, '2025-01-07'),
 (3, 8, 'East',  'Laptop',  1150.00, '2025-01-12'),
 (4, 4, 'East',  'Keyboard',  80.00, '2025-01-15'),
 (5, 6, 'West',  'Laptop',  1300.00, '2025-01-20'),
 (6, 8, 'North', 'Monitor',  280.00, '2025-02-02'),
 (7, 4, 'North', 'Laptop',  1250.00, '2025-02-03'),
 (8, 6, 'West',  'Keyboard',  95.00, '2025-02-09'),
 (9, 8, 'East',  'Monitor',  310.00, '2025-02-14'),
 (10,4, 'South', 'Laptop',  1180.00, '2025-02-18'),
 (11,6, 'South', 'Monitor',  295.00, '2025-02-21'),
 (12,8, 'North', 'Keyboard',  70.00, '2025-03-01'),
 (13,4, 'East',  'Laptop',  1220.00, '2025-03-04'),
 (14,6, 'West',  'Monitor',  320.00, '2025-03-08'),
 (15,8, 'South', 'Keyboard',  85.00, '2025-03-12');
''')
run("SELECT * FROM sales LIMIT 5")

,sale_id,employee_id,region,product,amount,sale_date
0,1,4,East,Laptop,1200.0,2025-01-05
1,2,6,West,Monitor,300.0,2025-01-07
2,3,8,East,Laptop,1150.0,2025-01-12
3,4,4,East,Keyboard,80.0,2025-01-15
4,5,6,West,Laptop,1300.0,2025-01-20


**Q1. Total revenue by region**, highest first.

In [14]:
run('''
SELECT region, SUM(amount) AS revenue
FROM sales
GROUP BY region
ORDER BY revenue ASC
''')

,region,revenue
0,South,1560.0
1,North,1600.0
2,West,2015.0
3,East,3960.0


<details>
<summary><b>💡 Reveal a possible solution</b> (try it yourself first!)</summary>

```python
run('''
SELECT region, SUM(amount) AS revenue
FROM sales
GROUP BY region
ORDER BY revenue DESC
''')
```

</details>

**Q2. Which product sold the most *units*** (number of sales rows)? Show the count for each product.

In [12]:
run('''
SELECT product, COUNT(*) AS units_sold
FROM sales
GROUP BY product
ORDER BY units_sold DESC
''')

,product,units_sold
0,Laptop,6
1,Monitor,5
2,Keyboard,4


<details>
<summary><b>💡 Reveal a possible solution</b> (try it yourself first!)</summary>

```python
run('''
SELECT product, COUNT(*) AS units_sold
FROM sales
GROUP BY product
ORDER BY units_sold DESC
''')
```

</details>

**Q3. The Top 3 single biggest sales:** show sale_id, region, product and amount.

In [ ]:
run('''
SELECT sale_id, region, product, amount
FROM sales
ORDER BY amount DESC
LIMIT 3
''')

,sale_id,region,product,amount
0,5,West,Laptop,1300.0
1,7,North,Laptop,1250.0
2,13,East,Laptop,1220.0


<details>
<summary><b>💡 Reveal a possible solution</b> (try it yourself first!)</summary>

```python
run('''
SELECT sale_id, region, product, amount
FROM sales
ORDER BY amount DESC
LIMIT 3
''')
```

</details>

**Q4. Laptop performance in the East and West only:** count and average laptop sale amount for each of those two regions.
*Hint: two conditions in `WHERE` joined with `AND`, plus `IN`.*

In [10]:
run('''
SELECT region, COUNT(*) AS laptops_sold, ROUND(AVG(amount), 2) AS avg_price
FROM sales
WHERE product = 'Laptop' AND region IN ('East', 'West')
GROUP BY region
''')

,region,laptops_sold,avg_price
0,East,3,1190.0
1,West,1,1300.0


<details>
<summary><b>💡 Reveal a possible solution</b> (try it yourself first!)</summary>

```python
run('''
SELECT region, COUNT(*) AS laptops_sold, ROUND(AVG(amount), 2) AS avg_price
FROM sales
WHERE product = 'Laptop' AND region IN ('East', 'West')
GROUP BY region
''')
```

</details>

**Q5. WHERE + HAVING together.** Considering only sales in **February or later** (`sale_date >= '2025-02-01'`), which regions brought in **more than $1,500**?

In [13]:
run('''
SELECT region, SUM(amount) AS revenue
FROM sales
WHERE sale_date >= '2025-02-01'
GROUP BY region
HAVING revenue > 1500
ORDER BY revenue DESC
''')

,region,revenue
0,North,1600.0
1,South,1560.0
2,East,1530.0


<details>
<summary><b>💡 Reveal a possible solution</b> (try it yourself first!)</summary>

```python
run('''
SELECT region, SUM(amount) AS revenue
FROM sales
WHERE sale_date >= '2025-02-01'
GROUP BY region
HAVING SUM(amount) > 1500
ORDER BY revenue DESC
''')
```

</details>

**✍️ Stretch thinking:** the `sales` table has an `employee_id` column that matches the Primary Key of `employees`. What do we call a column like that? *(This is your bridge to Session 4: JOINs!)*

<details>
<summary><b>💡 Reveal a sample answer</b></summary>

A **Foreign Key (FK)**: a column in one table that points to the Primary Key of another table. In Session 4 you will use `JOIN ... ON sales.employee_id = employees.employee_id` to combine the two tables.

</details>

---
## ✅ Lab Checklist
- [ ] Created a table with DDL and inserted rows with DML
- [ ] Wrote SELECT / WHERE / ORDER BY queries
- [ ] Summarized data with GROUP BY and aggregates
- [ ] Explained (and used) HAVING vs. WHERE
- [ ] Built a multi-clause query step by step
- [ ] Answered all five Regional Sales questions

## 💾 Save Your Work to GitHub
1. In Colab, click **File → Save a copy in GitHub**.
2. Choose your course/Capstone repository.
3. Use this commit message: `Session 3: Completed SQL Fundamentals Lab`

**Coming up next:** Session 4: JOINs, CASE WHEN, and subqueries in SQL Fiddle.